In [1]:
# train.py
import os
import json
from dataclasses import dataclass
from typing import List, Dict, Any
from functools import partial

import numpy as np
# from datasets import load_dataset, Dataset, concatenate_datasets
from torch.utils.data import Dataset 
from transformers import (
    AutoTokenizer,
    AutoModel,
    TrainingArguments,
    DataCollatorForLanguageModeling,
    Trainer
)
from transformers.trainer_utils import get_last_checkpoint
from accelerate import Accelerator
import torch
from sklearn.model_selection import train_test_split
# from tasks import render_example
import torch.nn as nn
import pandas as pd

import torch

from dataclasses import dataclass
from typing import Dict, Any, Callable, Optional
import evaluate
import shutil
import glob
from datetime import datetime

In [23]:
#data prep

odf = pd.read_csv("w22_entropy_differences_full.csv")
fdf = odf[["sentence_id", "text", "political_ideology", "label_binary", "simple_entropy"]]

#small for testing
# samped_sentences = fdf["sentence_id"].sample(n=2, random_state=42)
# fdf = fdf[fdf["sentence_id"].isin(samped_sentences)]  #

In [24]:
cat_cardinalities = {
    "label": 2,
    "words": 11267,
    "factual": 3,
    "group_id": 85,
    "type": 3,
    "topic": 14,
    "outlet": 8,
    "age": 54,
    "gender": 3,
    "education": 8,
    "native_english_speaker": 3,
    "political_ideology": 21,
    "followed_news_outlets": 551,
    "news_check_frequency": 6,
}

cat_embs_info = {}
for k, card in cat_cardinalities.items():
    if k not in fdf.columns:
        continue
    emb_dim = max(2, int(card ** 0.5) + 1) #sqrt(cardinality)) + 1
    emb_dim = min(50, emb_dim) #but not more than 50 - cough cough words
    cat_embs_info[k] = {"cardinality": card, "emb_dim": emb_dim}




class MTLDataset(Dataset):
    def __init__(self, df, cat_info, ambi_type, tokenizer, max_length=128):
        
        df = df.reset_index(drop=True).copy()
        self.texts = df["text"].tolist()
     
        for col in list(cat_info.keys()):
            df[col] = df[col].astype('category')
            df[col] = df[col].cat.codes

        cat_ids = df[list(cat_info.keys())].values   #[N, num_cat_vars]
        cat_ids = cat_ids.astype(int)

    
        self.cat_ids = cat_ids
        self.y_bias = df["label_binary"].to_numpy(dtype=np.float32)
        self.y_ambi = df[ambi_type].to_numpy(dtype=np.float32)
        self.tokenizer = tokenizer
        self.max_length = max_length

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        text = self.texts[idx]
        cats = self.cat_ids[idx]
        label_ambi = self.y_ambi[idx]
        label_bias = self.y_bias[idx]

        enc = self.tokenizer(
            text,
            padding="max_length",
            truncation=True,
            max_length=self.max_length,
            return_tensors="pt"
        )

        item = {
            "input_ids": enc["input_ids"].squeeze(0),
            "attention_mask": enc["attention_mask"].squeeze(0),
            "x_cats": torch.tensor(cats, dtype=torch.long),
            "y_bias": torch.tensor(label_bias, dtype=torch.float),
            "y_ambi": torch.tensor(label_ambi, dtype=torch.float),
        }
        return item


In [25]:
class MTLClassifier(nn.Module):
    def __init__(self, cats_info, llm_pretrained_name="meta-llama/Llama-2-7b-hf"):
        super().__init__()
        self.llm = AutoModel.from_pretrained(
            llm_pretrained_name,  
            output_hidden_states=True
        )

        for param in self.llm.parameters():
            param.requires_grad = False
            
        d_model = self.llm.config.hidden_size
        
#         self.cat_emb = nn.Embedding(num_cats, cat_emb_dim) #embedding table of the catvars
        cat_cards = [cats_info[c]["cardinality"] for c in cats_info.keys()]
        cat_emb_dim = [cats_info[c]["emb_dim"] for c in cats_info.keys()]
        self.cat_embs = nn.ModuleList([
            nn.Embedding(num_cats, emb_dim)
            for num_cats, emb_dim in zip(cat_cards, cat_emb_dim)
        ])
        
        late_fusion_width = d_model + (sum(cat_emb_dim))
        
        self.linear_post_fusion = nn.Linear(late_fusion_width, d_model)
        self.heads = nn.ModuleDict({
            "bias": nn.Sequential(
                nn.Linear(d_model, d_model // 2),
                nn.ReLU(),
                nn.Linear(d_model // 2, 1)          # bias prob and not bias prob
            ),
            "ambi": nn.Sequential(
                nn.Linear(d_model, d_model // 2),
                nn.ReLU(),
                nn.Linear(d_model // 2, 1),         # 1 logit
                nn.Sigmoid()                       # turn into probability
            )
        })

        self.loss_fct_ambi  = nn.MSELoss()
        self.loss_fct_bias = nn.BCEWithLogitsLoss()


    def forward(
            self, 
            input_ids, 
            attention_mask, 
            x_cats,              # [batch, n_cat_vars]
            y_bias=None,         # [batch] long
            y_ambi=None          # [batch] 0to1)
        ):


        #TEXT ENCODING
        outputs = self.llm(input_ids=input_ids, attention_mask=attention_mask)
        h_text = outputs.last_hidden_state[:, -1, :]  # [B, d_model]


        

        #CATS EMBEDDING
        cat_embs = []
        for i, emb_layer in enumerate(self.cat_embs):
            cat_i = x_cats[:, i]               # [batch]
            e_i = emb_layer(cat_i)            # [batch, emb_dim_i]
            cat_embs.append(e_i)
        e_cat = torch.cat(cat_embs, dim=-1)# [B, sum(emb_dims)]
        
        #GROUPED EMBEDDINGS
        h = torch.cat([h_text, e_cat], dim=-1)# [B, d_model + cat_emb_dim]
        h = torch.relu(self.linear_post_fusion(h))
        
        
        #HEADS
        logits_bias  = self.heads["bias"](h).squeeze(-1) # [B]
        logits_ambi = self.heads["ambi"](h).squeeze(-1) # [B]
        
        loss = None
        if y_bias is not None and y_ambi is not None:

            loss_bias = self.loss_fct_bias(logits_bias, y_bias)
            loss_ambi = self.loss_fct_ambi(logits_ambi,y_ambi)
            loss = loss_ambi + loss_bias
            
        return {
            "loss": loss,
            "logits_bias": logits_bias,
            "logits_ambi": logits_ambi
        }


In [26]:
#initzers
tokenizer = AutoTokenizer.from_pretrained("TinyLlama/TinyLlama-1.1B-Chat-v1.0", device_map="auto")

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token


model = MTLClassifier(
    cat_embs_info,
    "TinyLlama/TinyLlama-1.1B-Chat-v1.0"    
)
    
#datasets
ambi_type = "simple_entropy"
# x_fdf = fdf.drop(columns=["label_binary", ambi_type])
# y_fdf = fdf[["label_binary", ambi_type]]

# X_train, X_test, y_train, y_test = train_test_split(x_fdf, y_fdf, test_size=0.2, random_state=42)
train_df, test_df = train_test_split(fdf, test_size=0.2, random_state=42, shuffle=True)

train_dataset = MTLDataset(
    df=train_df,
    cat_info=cat_embs_info,
    ambi_type="simple_entropy",
    tokenizer=tokenizer,
    max_length=128,
)
    
eval_dataset = MTLDataset(
    df=test_df,
    cat_info=cat_embs_info,
    ambi_type="simple_entropy",
    tokenizer=tokenizer,
    max_length=128,
)


In [27]:
#training overhead
OUTPUT_DIR = f"./model/cv_results_{datetime.now().strftime('%Y%m%d_%H%M%S')}"
os.makedirs(OUTPUT_DIR, exist_ok=True)


training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=5e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,
    num_train_epochs=3,
    weight_decay=0.01,
    remove_unused_columns=False,
    logging_strategy="steps",
    logging_steps=50,
    # load_best_model_at_end=True,
    # metric_for_best_model="loss",
    # greater_is_better=False,
    save_total_limit=1,
    report_to=["none"],
    label_names=["y_bias", "y_ambi"]
)


### metrics
acc_metric = evaluate.load("accuracy")
f1_metric  = evaluate.load("f1")

def compute_metrics(eval_pred):
    # eval_pred: transformers.EvalPrediction
    # eval_pred.predictions: (bias_logits, ambi_logits)
    # eval_pred.label_ids: ground-truth labels

    logits_bias, logits_ambi = eval_pred.predictions
    logits_bias = np.asarray(logits_bias)
    logits_ambi = np.asarray(logits_ambi)
    
    labels_bias, labels_ambi = eval_pred.label_ids
    labels_bias = np.asarray(labels_bias)
    labels_ambi = np.asarray(labels_ambi)

    probs_bias = 1 / (1 + np.exp(-logits_bias))
    preds_bias = (probs_bias > 0.5).astype(int)

    #logits_ambi is like [0.69607705 0.7028114  0.68641293 0.6861405 ], so can just return
    preds_ambi = logits_ambi  #.argmax(axis=-1)

    


    #compel to agreeable 
    preds_bias  = np.asarray(preds_bias).reshape(-1)
    labels_bias = np.asarray(labels_bias).reshape(-1)
    preds_ambi  = np.asarray(preds_ambi).reshape(-1)
    labels_ambi = np.asarray(labels_ambi).reshape(-1)

    metrics = {}

    metrics.update({
        "bias_acc": acc_metric.compute(predictions=preds_bias, references=labels_bias)["accuracy"],
        "bias_f1": f1_metric.compute(predictions=preds_bias, references=labels_bias)["f1"],
    })

    #ambi metrics
    mse = np.mean((preds_ambi - labels_ambi)**2)
    rmse = np.sqrt(mse)
    
    metrics.update({
        "ambi_rmse": rmse
    })

    return metrics



trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    tokenizer=tokenizer,
    compute_metrics=compute_metrics
)



/scratch/slurm-1495812/ipykernel_3269142/879461188.py:78: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(


In [28]:
llama = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"
qwen_model_string = "Qwen/Qwen2-0.5B"


# ==== TRAIN ====
trainer.train()



Epoch,Training Loss,Validation Loss,Bias Acc,Bias F1,Ambi Rmse
1,0.669600,0.677834,0.668917,0.708447,0.265460
2,0.621900,0.642890,0.674262,0.721768,0.218266
3,0.594700,0.627718,0.673699,0.737912,0.196872


TrainOutput(global_step=2667, training_loss=0.6586753482625508, metrics={'train_runtime': 967.8282, 'train_samples_per_second': 44.078, 'train_steps_per_second': 2.756, 'total_flos': 0.0, 'train_loss': 0.6586753482625508, 'epoch': 3.0})

In [29]:
# ==== PREDS ====
test_pred = trainer.predict(eval_dataset)
preds = test_pred.predictions

# ### bias performance
logits_bias, logits_ambi = preds
labels_bias, labels_ambi = test_pred.label_ids

probs_bias = 1 / (1 + np.exp(-logits_bias))
preds_bias = (probs_bias > 0.5).astype(int)

metrics = {}

metrics.update({
    "bias_acc": acc_metric.compute(predictions=preds_bias, references=labels_bias)["accuracy"],
    "bias_f1": f1_metric.compute(predictions=preds_bias, references=labels_bias)["f1"],
})

mse = np.mean((logits_ambi - labels_ambi)**2)
rmse = np.sqrt(mse)

metrics.update({
    "ambi_rmse": rmse
})


In [30]:
metrics

{'bias_acc': 0.6736990154711674,
 'bias_f1': 0.7379123361952101,
 'ambi_rmse': np.float32(0.19687183)}

In [20]:
print(logits_bias)
print(logits_ambi)
print(labels_bias)
print(labels_ambi)
print(preds_bias)

[0.7211827  0.71713936 0.6625634  0.67527497]
[0.819284   0.8179361  0.81056607 0.81006736]
[0. 0. 0. 1.]
[1.        1.        0.9709506 0.9709506]
[1 1 1 1]


In [ ]:
# FINAL SAVE
final_dir = OUTPUT_DIR + "/mt_model_final"

os.makedirs(final_dir, exist_ok=True)

torch.save(model.state_dict(), f"{final_dir}/pytorch_model.bin")
shutil.make_archive("mt_model_final", "zip", final_dir)

In [ ]:
full_dataset = MTLDataset(
    df=fdf,
    cat_info=cat_embs_info,
    ambi_type="simple_entropy",
    tokenizer=tokenizer,
    max_length=128,
)


test_pred = trainer.predict(full_dataset)
preds = test_pred.predictions

# ### bias performance
logits_bias, logits_ambi = preds
labels_bias, labels_ambi = test_pred.label_ids

probs_bias = 1 / (1 + np.exp(-logits_bias))
preds_bias = (probs_bias > 0.5).astype(int)

fdff = fdf.copy()
fdff["modA_bias_pred"] = preds_bias
fdff["modA_ambi_pred"] = logits_ambi

fdff.to_csv("modA_pred_scores")

# JUNK